In [3]:
pip install yfinance pandas numpy

In [4]:
# If needed (in a fresh environment):
# !pip -q install yfinance pandas numpy

from __future__ import annotations

import os
from pathlib import Path
import pandas as pd
import numpy as np
import yfinance as yf

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

TICKER_MAP_CSV = DATA_DIR / "ticker_map.csv"
PRICES_CSV     = DATA_DIR / "prices_adjclose_daily.csv"
RETURNS_CSV    = DATA_DIR / "returns_weekly.csv"

# Choose a weekly anchor. In many quant workflows, Friday close is used.
WEEKLY_FREQ = "W-FRI"


In [5]:
assets = [
    # Equities - US core
    {"asset_id":"US_EQ_SPY", "yfticker":"SPY", "asset_name":"SPDR S&P 500", "asset_class":"Equity", "region":"US", "ccy":"USD"},
    {"asset_id":"US_EQ_QQQ", "yfticker":"QQQ", "asset_name":"Invesco QQQ (Nasdaq 100)", "asset_class":"Equity", "region":"US", "ccy":"USD"},
    {"asset_id":"US_EQ_IWM", "yfticker":"IWM", "asset_name":"iShares Russell 2000", "asset_class":"Equity", "region":"US", "ccy":"USD"},
    {"asset_id":"US_EQ_VTV", "yfticker":"VTV", "asset_name":"Vanguard Value", "asset_class":"Equity", "region":"US", "ccy":"USD"},

    # Equities - ex US
    {"asset_id":"EU_EQ_IEV", "yfticker":"IEV", "asset_name":"iShares MSCI Europe", "asset_class":"Equity", "region":"Europe", "ccy":"USD"},
    {"asset_id":"JP_EQ_EWJ", "yfticker":"EWJ", "asset_name":"iShares MSCI Japan", "asset_class":"Equity", "region":"Japan", "ccy":"USD"},
    {"asset_id":"EM_EQ_EEM", "yfticker":"EEM", "asset_name":"iShares MSCI Emerging Markets", "asset_class":"Equity", "region":"EM", "ccy":"USD"},

    # Bonds
    {"asset_id":"US_BD_IEF", "yfticker":"IEF", "asset_name":"iShares 7-10Y Treasuries", "asset_class":"Bond", "region":"US", "ccy":"USD"},
    {"asset_id":"US_BD_TLT", "yfticker":"TLT", "asset_name":"iShares 20+Y Treasuries", "asset_class":"Bond", "region":"US", "ccy":"USD"},
    {"asset_id":"US_BD_TIP", "yfticker":"TIP", "asset_name":"iShares TIPS", "asset_class":"Bond", "region":"US", "ccy":"USD"},

    # Commodities
    {"asset_id":"CMD_GLD", "yfticker":"GLD", "asset_name":"SPDR Gold Shares", "asset_class":"Commodity", "region":"Global", "ccy":"USD"},
    {"asset_id":"CMD_USO", "yfticker":"USO", "asset_name":"United States Oil Fund", "asset_class":"Commodity", "region":"Global", "ccy":"USD"},
    {"asset_id":"CMD_DBC", "yfticker":"DBC", "asset_name":"Invesco DB Commodity Index", "asset_class":"Commodity", "region":"Global", "ccy":"USD"},

    # FX (Yahoo convention)
    {"asset_id":"FX_EURUSD", "yfticker":"EURUSD=X", "asset_name":"EUR/USD", "asset_class":"FX", "region":"G10", "ccy":"USD"},
    {"asset_id":"FX_JPYUSD", "yfticker":"JPYUSD=X", "asset_name":"JPY/USD", "asset_class":"FX", "region":"G10", "ccy":"USD"},
]

ticker_map = pd.DataFrame(assets)
ticker_map.to_csv(TICKER_MAP_CSV, index=False)
ticker_map


,asset_id,yfticker,asset_name,asset_class,region,ccy
0,US_EQ_SPY,SPY,SPDR S&P 500,Equity,US,USD
1,US_EQ_QQQ,QQQ,Invesco QQQ (Nasdaq 100),Equity,US,USD
2,US_EQ_IWM,IWM,iShares Russell 2000,Equity,US,USD
3,US_EQ_VTV,VTV,Vanguard Value,Equity,US,USD
4,EU_EQ_IEV,IEV,iShares MSCI Europe,Equity,Europe,USD
5,JP_EQ_EWJ,EWJ,iShares MSCI Japan,Equity,Japan,USD
6,EM_EQ_EEM,EEM,iShares MSCI Emerging Markets,Equity,EM,USD
7,US_BD_IEF,IEF,iShares 7-10Y Treasuries,Bond,US,USD
8,US_BD_TLT,TLT,iShares 20+Y Treasuries,Bond,US,USD
9,US_BD_TIP,TIP,iShares TIPS,Bond,US,USD


In [6]:
def load_ticker_map(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    required = {"asset_id","yfticker","asset_name","asset_class","region","ccy"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in ticker map: {missing}"   )